In [ ]:
#### import meci_geoms, perform graph analysis on optimized meci_geom, outputs csv with column 1: name of meci input, column 2: class of meci based off rmsd threshold equivlancy
# If dissociation, bin seperately due to opimization probelms with dissociation. perform sorting of MECIs and assignment 

In [2]:
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds
import numpy as np
import pandas as pd 
from scipy.spatial.transform import Rotation as R
from scipy.spatial.distance import pdist, squareform
from pathlib import Path 
import tqdm
import plotly.express as px
import plotly.subplots as sp
import umap
from sklearn.cluster import KMeans
import itertools


In [ ]:
def xyz_to_rdkit_mol(xyz_file, total_charge):
    """
    Convert an XYZ file with explicit hydrogens to an RDKit Mol,
    preserving 3D coordinates and determining connectivity automatically.
    

    Parameters:
        xyz_file: path to XYZ file
        total_charge: total molecular charge (default 0)
    
    Returns:
        RDKit Mol object
    """
    # --- Read XYZ ---
    with open(xyz_file) as f:
        lines = f.readlines()[2:]  # skip atom count + comment
    atoms, coords = [], []
    for line in lines:
        parts = line.split()
        atoms.append(parts[0])
        coords.append([float(x) for x in parts[1:4]])
    coords = np.array(coords)
    
    # --- Create empty molecule with atoms ---
    mol = Chem.RWMol()
    z = [Chem.GetPeriodicTable().GetAtomicNumber(a) for a in atoms]
    for Zi in z:
        mol.AddAtom(Chem.Atom(Zi))
    
    # --- Add coordinates ---
    conf = Chem.Conformer(len(coords))
    for i, pos in enumerate(coords):
        conf.SetAtomPosition(i, pos)
    mol.AddConformer(conf)
    
    # --- Determine connectivity using RDKit's bond perception ---
    Chem.rdDetermineBonds.DetermineConnectivity(mol, charge=total_charge)
    
    # --- Sanitize molecule ---
    Chem.SanitizeMol(mol)
    
    return mol



def align_and_rmsd_numpy(X, Y, return_aligned=False):
    X_center = X.mean(axis=0)
    Y_center = Y.mean(axis=0)
    Xc = X - X_center
    Yc = Y - Y_center

    rot, _ = R.align_vectors(Xc, Yc)
    Y_aligned = rot.apply(Yc) + X_center
    rmsd = np.sqrt(np.mean(np.sum((X - Y_aligned)**2, axis=1)))

    if return_aligned:
        return rmsd, Y_aligned
    else:
        return rmsd


def update_mol_coordinates_copy(mol, new_coords):
    """
    Return a copy of an RDKit molecule with updated 3D coordinates.
    
    Parameters:
        mol: RDKit Mol object (must have same number of atoms as new_coords)
        new_coords: (N,3) numpy array of new coordinates
    
    Returns:
        new_mol: RDKit Mol object copy with updated coordinates
    """
    if new_coords.shape[0] != mol.GetNumAtoms():
        raise ValueError("Number of coordinates must match number of atoms")
    
    # --- Make a true copy ---
    new_mol = Chem.Mol(mol)  # copy molecule
    # Remove old conformers
    for conf_id in [c.GetId() for c in new_mol.GetConformers()]:
        new_mol.RemoveConformer(conf_id)
    
    # --- Add new conformer ---
    conf = Chem.Conformer(new_mol.GetNumAtoms())
    for i, pos in enumerate(new_coords):
        conf.SetAtomPosition(i, pos)
    new_mol.AddConformer(conf)
    
    return new_mol

def reflect_axis(geometry, reflection): 
    reflected_geom = geometry * reflection 
    return reflected_geom

def numpy_geom(mol):
    conf = mol.GetConformer()
    return np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])


REFLECTIONS = np.array(
    [
        [1, 1, 1],
        [-1, 1, 1],
        [1, -1, 1],
        [1, 1, -1],
        [-1, 1, -1],
        [-1, -1, 1],
        [1, -1, -1],
        [-1, -1, -1],
    ]
)


def check_identity(X, Y, use_reflections = True, threshold = 0.05): 
    X_geom = numpy_geom(X)
    Y_geom = numpy_geom(Y)
    identical = False

    matches = Y.GetSubstructMatches(X, uniquify=False)
    rmsds = []
    for match in matches:
        #print(match)
        #print(Y_geom)
        swapped = Y_geom[list(match)]  
        #print(swapped)


            
        if use_reflections == True: 
            for reflection in REFLECTIONS: 
                swapped_reflection = reflect_axis(swapped, reflection)
                rmsd, aligned_coords = align_and_rmsd_numpy(X_geom, swapped_reflection, return_aligned=True)
                rmsds.append((rmsd, aligned_coords))


        else:


           rmsd, aligned_coords = align_and_rmsd_numpy(X_geom, swapped, return_aligned=True)
           rmsds.append((rmsd, aligned_coords))

    best_rmsd, best_aligned = min(rmsds, key=lambda t: t[0])
    #print(best_rmsd)
    if best_rmsd < threshold :
        identical = True

    return identical, best_rmsd





In [ ]:
meci_geom_folder = Path('../data/raw_geometries/meci/benzene/')



In [ ]:
labels = {}
unique_geoms = {}

for x in tqdm.tqdm(meci_geom_folder.glob('*')):
    test_mol = xyz_to_rdkit_mol(x, total_charge=0)
    smiles = Chem.MolToSmiles(test_mol, canonical=True)

    if smiles not in unique_geoms:
        unique_geoms[smiles] = [x]
        continue

    #Handle disconnected systems
    if "." in smiles:
        labels[x.stem] = unique_geoms[smiles][0].stem
        continue

    found_identical = False

    for template in unique_geoms[smiles]:
        template_mol = xyz_to_rdkit_mol(template, total_charge=0)

        identical, best_rmsd = check_identity(template_mol, test_mol)

        if identical:
            labels[x.stem] = template.stem
            found_identical = True
            break

    if not found_identical:
        unique_geoms[smiles].append(x)


1183it [00:21, 54.41it/s]


In [13]:
from collections import Counter

label_counts = Counter(labels.values())

for label, count in label_counts.items():
    print(label, count)

0081_15 421
0070_13 343
0178_7 386
0146_13 2
0144_11 7
0079_8 6
0082_4 2
0150_15 3
0013_14 1


In [ ]:
#print(labels)

{'0435_3': '0470_3', '0572_4': '0470_3', '0022_5': '0470_3', '0861_5': '0470_3', '0318_3': '0648_2', '0926_2': '0120_2', '0198_2': '0470_3', '0771_3': '0120_2', '0549_2': '0470_3', '0734_3': '0470_3', '0636_4': '0470_3', '0673_4': '0470_3', '0221_2': '0470_3', '0019_3': '0120_2', '0264_2': '0120_2', '0013_13': '0299_2', '0513_5': '0470_3', '0187_5': '0470_3', '0454_2': '0470_3', '0085_2': '0470_3', '0411_2': '0120_2', '0629_3': '0470_3', '0947_3': '0120_2', '0104_3': '0120_2', '0902_3': '0648_2', '0141_3': '0648_2', '0590_3': '0120_2', '0379_2': '0120_2', '0845_4': '0120_2', '0006_4': '0470_3', '0612_5': '0470_3', '0384_2': '0120_2', '0755_2': '0470_3', '0710_2': '0299_2', '0205_3': '0120_2', '0078_2': '0470_3', '0691_3': '0120_2', '0240_3': '0470_3', '0939_5': '0470_3', '0307_4': '0470_3', '0885_4': '0470_3', '0181_3': '0470_3', '0768_2': '0120_2', '0550_3': '0470_3', '0494_2': '0120_2', '0806_2': '0120_2', '0045_2': '0120_2', '0238_3': '0120_2', '0843_2': '0648_2', '0000_2': '0120_2'

In [8]:
df = pd.DataFrame(labels.items(), columns = ['meci','meci_type'])

In [ ]:
#print(df)

df.to_csv('../data/meci_classification/benzene', index=False)

         meci meci_type
0      0041_6   0081_15
1     0013_13   0070_13
2     0098_20   0070_13
3      0096_8   0081_15
4     0148_17   0070_13
...       ...       ...
1167   0067_8   0081_15
1168  0051_13   0070_13
1169  0151_13   0070_13
1170  0104_12   0081_15
1171  0086_11   0081_15

[1172 rows x 2 columns]


In [139]:
######## DEBUG


mol1 = xyz_to_rdkit_mol('/Users/connerbaucom/Desktop/Pieri/CTG/master_graphs/ethylene/data_1.0/MECI/0321_4.xyz', total_charge=0)
mol2 = xyz_to_rdkit_mol('/Users/connerbaucom/Desktop/Pieri/CTG/master_graphs/ethylene/data_1.0/MECI/0120_2.xyz', total_charge=0)

check_identity(mol1,mol2)




(False, np.float64(0.44628530540682976))